# Demo A — Da PCAP a Grafo di Rete
**Master Cybersecurity LUISS — 29 maggio 2026**  
*Docente: Antonio Formato*

---

## Obiettivo didattico

Questo notebook dimostra come trasformare una cattura di traffico di rete (PCAP) in un **grafo diretto pesato**, applicando poi algoritmi di teoria dei grafi per estrarre insight operativi di sicurezza.

**Pipeline completa:** `tshark` (estrazione) → `pandas` (manipolazione tabulare) → `networkx` (modello grafo) → `D3.js` (visualizzazione interattiva)

### Viste progressive sullo stesso dataset

| # | Vista | Scopo pedagogico |
|---|-------|------------------|
| 1 | **Naive** | Tutti i flussi, nessuna aggregazione — dimostra il *caos visivo* e la necessità di preprocessing |
| 2 | **Aggregata pesata** | Stessi nodi, archi pesati per volume — gli hub emergono visivamente |
| 3 | **Filtrata (external only)** | Solo flussi interno→esterno — evidenzia potenziali C2/exfiltration |

### Collegamento con la teoria (T2)

Nella sezione bonus applichiamo gli stessi algoritmi trattati nella lezione teorica T2 (PageRank, componenti connesse) al grafo costruito dal pcap reale, chiudendo il cerchio teoria→pratica.

## Cella 1 — Setup dell'ambiente

### Dipendenze Python
```bash
pip install pandas networkx pyvis
```

### Requisito esterno: `tshark`
`tshark` è l'interfaccia CLI di Wireshark. Ci permette di estrarre campi specifici dai pacchetti senza GUI, operazione fondamentale in pipeline automatizzate di threat hunting.

| OS | Installazione |
|----|---------------|
| Windows | Incluso con Wireshark → aggiungere `C:\Program Files\Wireshark` al PATH |
| macOS | `brew install wireshark` |
| Ubuntu/Debian | `sudo apt install tshark` |

> **Nota:** se `tshark` non è disponibile nel vostro ambiente, più sotto è presente un *fallback* che carica un CSV pre-estratto.

In [21]:
import subprocess
import ipaddress
from pathlib import Path

import pandas as pd
import networkx as nx
from pyvis.network import Network

In [22]:
# ───── CONFIGURAZIONE ─────
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
FLOWS_CSV = OUTPUT_DIR / "flows.csv"
PCAP_DIR = Path("pcaps")

# ─── Scansione PCAP disponibili ───
pcap_files = sorted(PCAP_DIR.glob("*.pcap")) + sorted(PCAP_DIR.glob("*.pcapng"))

if not pcap_files:
    print("⚠ Nessun file pcap trovato in", PCAP_DIR)
    print("  → Verrà usato il CSV pre-estratto (output/flows.csv) se disponibile.")
    PCAP = None
else:
    print("=" * 70)
    print("  PCAP DISPONIBILI PER L'ANALISI")
    print("=" * 70)
    for i, p in enumerate(pcap_files, 1):
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"  [{i}] {p.name}")
        print(f"      ({size_mb:.1f} MB)")
    print("-" * 70)
    print(f"  [0] Salta estrazione — usa CSV pre-estratto (output/flows.csv)")
    print("=" * 70)

    # Per esecuzione automatica: usa CSV pre-estratto (opzione 0)
    # Per esecuzione interattiva decommentare la riga seguente:
    # scelta = input("\n→ Scegli il numero del pcap da analizzare: ").strip()
    scelta = "0"

    if scelta == "0" or scelta == "":
        print("\n✓ Uso il CSV pre-estratto esistente.")
        PCAP = None
    elif scelta.isdigit() and 1 <= int(scelta) <= len(pcap_files):
        PCAP = str(pcap_files[int(scelta) - 1])
        print(f"\n✓ Selezionato: {PCAP}")
    else:
        print(f"\n⚠ Scelta non valida '{scelta}'. Uso il CSV pre-estratto.")
        PCAP = None

  PCAP DISPONIBILI PER L'ANALISI
  [1] 2026-02-03-GuLoader-for-AgentTesla-style-infection-with-FTP-data-exfil.pcap
      (0.3 MB)
  [2] 2026-05-11-macOS-malware-infection-traffic.pcap
      (18.2 MB)
----------------------------------------------------------------------
  [0] Salta estrazione — usa CSV pre-estratto (output/flows.csv)

✓ Uso il CSV pre-estratto esistente.


## Cella 3 — Estrazione flussi con `tshark`

### Selezione interattiva
La cella precedente presenta un **menu di scelta** dei pcap disponibili nella cartella `pcaps/`. L'utente seleziona quale cattura analizzare oppure sceglie l'opzione `[0]` per utilizzare un CSV pre-estratto.

### Razionale
Il primo passo di qualsiasi analisi network forense è l'**estrazione strutturata** dei metadati dai pacchetti. Non ci interessa il payload (che richiederebbe DPI), ma la *tupla di connessione*: `(IP_src, IP_dst, porta_dst)`.

### Campi estratti
| Campo tshark | Semantica |
|--------------|-----------|
| `ip.src` | Indirizzo IP sorgente (layer 3) |
| `ip.dst` | Indirizzo IP destinazione (layer 3) |
| `tcp.dstport` | Porta destinazione TCP (layer 4) |
| `udp.dstport` | Porta destinazione UDP (layer 4) |

### Filtro applicato
`-Y ip` → solo pacchetti con header IPv4 valido (esclude ARP, STP, etc.)

### PCAP inclusi nel repository
| # | File | Scenario |
|---|------|----------|
| 1 | `2026-02-03-GuLoader-for-AgentTesla-style-infection-with-FTP-data-exfil.pcap` | Infezione Windows: loader → RAT → exfiltration FTP |
| 2 | `2026-05-11-macOS-malware-infection-traffic.pcap` | Infezione macOS: analisi traffico post-compromise |

> **In produzione:** questa estrazione è il primo stadio di pipeline SOAR/SIEM che alimentano grafi di comportamento per anomaly detection.

In [23]:
def extract_flows(pcap_path, csv_path):
    """Estrae src/dst/porte da un pcap usando tshark."""
    cmd = [
        "tshark", "-r", str(pcap_path),
        "-T", "fields",
        "-e", "ip.src",
        "-e", "ip.dst",
        "-e", "tcp.dstport",
        "-e", "udp.dstport",
        "-E", "separator=,",
        "-E", "header=n",
        "-Y", "ip",   # solo pacchetti con IP
    ]
    with open(csv_path, "w") as f:
        subprocess.run(cmd, stdout=f, check=True)
    print(f"✓ Estratto in {csv_path}")

# ─── Esegui estrazione solo se un PCAP è stato selezionato ───
if PCAP is not None:
    extract_flows(PCAP, FLOWS_CSV)
    print(f"  Sorgente: {PCAP}")
    print(f"  Output:   {FLOWS_CSV}")
elif FLOWS_CSV.exists():
    print(f"✓ Estrazione saltata — uso CSV esistente: {FLOWS_CSV}")
    print(f"  ({FLOWS_CSV.stat().st_size / 1024:.0f} KB)")
else:
    raise FileNotFoundError(
        f"Nessun PCAP selezionato e {FLOWS_CSV} non esiste.\n"
        "Torna alla cella precedente e scegli un pcap, oppure posiziona un CSV in output/flows.csv"
    )

✓ Estrazione saltata — uso CSV esistente: output\flows.csv
  (670 KB)


### Cella 4 — Fallback: CSV pre-estratto

Se `tshark` non è disponibile nell'ambiente corrente, il CSV risultante è già presente in `output/flows.csv`. Il comando equivalente da terminale sarebbe:

```bash
tshark -r sample.pcap -T fields -e ip.src -e ip.dst -e tcp.dstport -e udp.dstport -E separator=, -Y ip > output/flows.csv
```

In alternativa, la cella successiva contiene un generatore di **dati sintetici** (commentato) utile per testing senza pcap reali.

In [ ]:
# # Dati sintetici di prova — decommentare per testare senza pcap
# import random
# random.seed(42)
# internal_hosts = [f"10.0.0.{i}" for i in range(1, 8)]
# external_hosts = [f"203.0.113.{i}" for i in [10, 25, 42, 77]] + ["8.8.8.8"]
# c2 = "203.0.113.42"  # nostro "C2" finto: beaconing
# rows = []
# for _ in range(500):
#     src = random.choice(internal_hosts)
#     dst = random.choice(external_hosts + internal_hosts)
#     dport = random.choice([80, 443, 53, 22, 3389])
#     rows.append((src, dst, dport, ""))
# # beaconing fitto verso il C2
# for _ in range(200):
#     rows.append((random.choice(internal_hosts), c2, 443, ""))
# pd.DataFrame(rows, columns=["src","dst","tcp_dport","udp_dport"]).to_csv(FLOWS_CSV, index=False, header=False)
# print("✓ CSV sintetico creato")

## Cella 5 — Caricamento dati e ispezione esplorativa

### Obiettivo
Caricare il CSV in un `DataFrame` pandas e ottenere una prima caratterizzazione statistica del dataset:
- **Cardinalità** del traffico (numero flussi totali)
- **Unicità** degli endpoint (quanti IP distinti partecipano)
- **Diversità** delle porte di destinazione (indicatore di volume di servizi contattati)

### Perché è rilevante in cybersecurity
Un singolo host che contatta *molte* porte diverse su *molti* IP esterni è un pattern classico di **port scanning**, **beaconing** verso infrastruttura C2, o **data exfiltration** distribuita. Questi numeri ci danno un primo "odore" del traffico prima ancora di costruire il grafo.

In [24]:
df = pd.read_csv(FLOWS_CSV, names=["src", "dst", "tcp_dport", "udp_dport"], dtype=str)
df["dport"] = df["tcp_dport"].fillna(df["udp_dport"])
df = df.dropna(subset=["src", "dst"])

print(f"Flussi totali:      {len(df):,}")
print(f"IP src unici:       {df['src'].nunique()}")
print(f"IP dst unici:       {df['dst'].nunique()}")
print(f"Porte dst uniche:   {df['dport'].nunique()}")
df.head()

Flussi totali:      20,555
IP src unici:       38
IP dst unici:       42
Porte dst uniche:   181


,src,dst,tcp_dport,udp_dport,dport
0,10.5.11.101,10.5.11.1,NaN,53,53
1,10.5.11.101,10.5.11.1,NaN,53,53
2,10.5.11.1,10.5.11.101,NaN,62429,62429
3,10.5.11.1,10.5.11.101,NaN,50153,50153
4,10.5.11.101,142.250.114.113,443,NaN,443


## Cella 6 — Vista 1: Grafo ingenuo (naive)

### Definizione formale
Costruiamo un **digrafo** $G_1 = (V, E)$ dove:
- $V$ = insieme di tutti gli indirizzi IP (sorgente e destinazione)
- $E$ = insieme di archi diretti, con $(u, v) \in E$ sse esiste almeno un pacchetto con `ip.src = u` e `ip.dst = v`

### Scopo pedagogico
Questa vista **non** è utile operativamente. Serve a dimostrare visivamente che:
1. Un grafo non processato è **illeggibile** (information overload)
2. L'**aggregazione** e il **filtraggio** sono prerequisiti per l'analisi visuale
3. Il *raw graph* nasconde la struttura — che invece emergerà nelle viste successive

> **Analogia accademica:** è lo stesso principio per cui in statistica si parte dai dati grezzi per poi calcolare indicatori sintetici. Il grafo naive è il "dump tabellare", le viste successive sono le "statistiche descrittive".

In [25]:
G1 = nx.from_pandas_edgelist(df, "src", "dst", create_using=nx.DiGraph())
print(f"Nodi: {G1.number_of_nodes()}, Archi: {G1.number_of_edges()}")

net1 = Network(notebook=False, directed=True, height="650px", width="100%",
               bgcolor="#1a1a1a", font_color="white")
net1.from_nx(G1)
net1.write_html(str(OUTPUT_DIR / "view1_naive.html"), open_browser=False, notebook=False)
print(f"→ {OUTPUT_DIR / 'view1_naive.html'}")

Nodi: 42, Archi: 78
→ output\view1_naive.html


## Cella 7 — Vista 2: Grafo aggregato e pesato

### Definizione formale
Costruiamo un **digrafo pesato** $G_2 = (V, E, w)$ dove:
- $V$ = stessi nodi di $G_1$
- $E$ = stessi archi di $G_1$ (una sola coppia per ogni combinazione src→dst)
- $w: E \to \mathbb{N}$ = funzione peso, dove $w(u,v)$ è il **numero totale di pacchetti** da $u$ a $v$

### Trasformazione applicata
```
GROUP BY (src, dst) → COUNT(*) AS weight
```
Questa aggregazione comprime migliaia di record in decine di archi pesati, rivelando la **struttura latente**.

### Encoding visuale (D3.js)
| Proprietà visiva | Variabile codificata | Scala |
|------------------|---------------------|-------|
| Raggio nodo | Volume totale (in + out) | $\sqrt{\cdot}$, range 4–28px |
| Colore nodo | Classificazione hub/mid/leaf | Categorica (arancio/blu/grigio) |
| Spessore arco | Peso $w(e)$ | $\text{pow}(0.5)$, range 0.3–3.5px |
| Opacità arco | Peso $w(e)$ | $\text{pow}(0.7)$, range 15%–70% |

### Cosa osservare
L'**hub emergente** (nodo arancione, grande) è l'host con il maggior volume di traffico — nel contesto di un'infezione, è il paziente zero.

In [26]:
import math, json

weighted = df.groupby(["src", "dst"]).size().reset_index(name="weight")

# ─── Costruzione grafo NetworkX G2 (usato anche nelle celle successive) ───
G2 = nx.from_pandas_edgelist(weighted, "src", "dst", edge_attr="weight", create_using=nx.DiGraph())
print(f"G2: {G2.number_of_nodes()} nodi, {G2.number_of_edges()} archi")

# ─── Calcolo metriche per nodi ───
out_volume = weighted.groupby("src")["weight"].sum().to_dict()
in_volume = weighted.groupby("dst")["weight"].sum().to_dict()
all_ips = set(weighted["src"]) | set(weighted["dst"])
max_volume = max(out_volume.get(n, 0) + in_volume.get(n, 0) for n in all_ips)
max_weight = weighted["weight"].max()

# ─── Preparazione dati JSON per D3 ───
nodes_data = []
for n in all_ips:
    total = out_volume.get(n, 0) + in_volume.get(n, 0)
    norm = math.log1p(total) / math.log1p(max_volume)
    # Classificazione: hub / medio / periferia
    if norm > 0.7:
        group = "hub"
    elif norm > 0.25:
        group = "mid"
    else:
        group = "leaf"
    nodes_data.append({
        "id": n,
        "volume": total,
        "norm": round(norm, 4),
        "group": group
    })

links_data = []
for _, row in weighted.iterrows():
    w = row["weight"]
    norm_w = math.log1p(w) / math.log1p(max_weight)
    links_data.append({
        "source": row["src"],
        "target": row["dst"],
        "weight": int(w),
        "norm": round(norm_w, 4)
    })

graph_json = json.dumps({"nodes": nodes_data, "links": links_data})

# ─── Template HTML con D3.js ───
html_view2 = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>Vista 2 — Grafo Aggregato e Pesato</title>
<script src="https://d3js.org/d3.v7.min.js"></script>
<style>
  body {{ margin: 0; background: #0d1117; overflow: hidden; font-family: 'JetBrains Mono', 'Fira Code', monospace; }}
  svg {{ display: block; }}
  .tooltip {{
    position: absolute; padding: 8px 12px; background: rgba(13,17,23,0.95);
    border: 1px solid #30363d; border-radius: 6px; color: #e6edf3;
    font-size: 12px; pointer-events: none; opacity: 0; transition: opacity 0.2s;
    box-shadow: 0 4px 12px rgba(0,0,0,0.4);
  }}
  .legend {{
    position: absolute; top: 16px; left: 16px; background: rgba(13,17,23,0.9);
    border: 1px solid #30363d; border-radius: 8px; padding: 14px 18px;
    color: #e6edf3; font-size: 12px;
  }}
  .legend h3 {{ margin: 0 0 10px 0; font-size: 14px; color: #58a6ff; }}
  .legend-item {{ display: flex; align-items: center; margin: 5px 0; }}
  .legend-dot {{ width: 12px; height: 12px; border-radius: 50%; margin-right: 8px; border: 2px solid rgba(255,255,255,0.3); }}
  .stats {{
    position: absolute; bottom: 16px; right: 16px; background: rgba(13,17,23,0.9);
    border: 1px solid #30363d; border-radius: 8px; padding: 12px 16px;
    color: #8b949e; font-size: 11px;
  }}
</style></head><body>
<div class="legend">
  <h3>Vista 2 — Grafo Pesato</h3>
  <div class="legend-item"><div class="legend-dot" style="background:#f0883e"></div>Hub (alto volume)</div>
  <div class="legend-item"><div class="legend-dot" style="background:#58a6ff"></div>Intermedio</div>
  <div class="legend-item"><div class="legend-dot" style="background:#8b949e"></div>Periferia</div>
  <div style="margin-top:10px; color:#8b949e; font-size:11px;">
    Spessore arco = volume flussi<br>
    Dimensione nodo = traffico totale
  </div>
</div>
<div class="stats">
  Nodi: {len(nodes_data)} &nbsp;|&nbsp; Archi: {len(links_data)} &nbsp;|&nbsp; Max peso: {max_weight:,}
</div>
<div class="tooltip" id="tooltip"></div>
<script>
const data = {graph_json};
const width = window.innerWidth, height = window.innerHeight;

const colorMap = {{ hub: "#f0883e", mid: "#58a6ff", leaf: "#8b949e" }};
const radiusScale = d3.scaleSqrt().domain([0, 1]).range([4, 28]);
const widthScale = d3.scalePow().exponent(0.5).domain([0, 1]).range([0.3, 3.5]);
const opacityScale = d3.scalePow().exponent(0.7).domain([0, 1]).range([0.15, 0.7]);

const svg = d3.select("body").append("svg")
  .attr("width", width).attr("height", height);

// Arrow marker (piccolo, fisso, non scala con width)
svg.append("defs").append("marker")
  .attr("id", "arrow").attr("viewBox", "0 -3 6 6")
  .attr("refX", 20).attr("refY", 0)
  .attr("markerWidth", 5).attr("markerHeight", 5)
  .attr("orient", "auto")
  .append("path").attr("d", "M0,-3L6,0L0,3")
  .attr("fill", "#484f58");

const simulation = d3.forceSimulation(data.nodes)
  .force("link", d3.forceLink(data.links).id(d => d.id).distance(120))
  .force("charge", d3.forceManyBody().strength(-300))
  .force("center", d3.forceCenter(width/2, height/2))
  .force("collision", d3.forceCollide().radius(d => radiusScale(d.norm) + 5));

const link = svg.append("g").selectAll("line").data(data.links).join("line")
  .attr("stroke", d => d3.interpolateRgb("#30363d", "#58a6ff")(d.norm))
  .attr("stroke-width", d => widthScale(d.norm))
  .attr("stroke-opacity", d => opacityScale(d.norm))
  .attr("marker-end", "url(#arrow)");

const node = svg.append("g").selectAll("g").data(data.nodes).join("g")
  .call(d3.drag().on("start", dragstarted).on("drag", dragged).on("end", dragended));

node.append("circle")
  .attr("r", d => radiusScale(d.norm))
  .attr("fill", d => colorMap[d.group])
  .attr("stroke", d => d3.color(colorMap[d.group]).brighter(0.5))
  .attr("stroke-width", 1.5)
  .attr("opacity", 0.9);

node.append("text")
  .text(d => d.id)
  .attr("dx", d => radiusScale(d.norm) + 4)
  .attr("dy", 3)
  .attr("fill", "#e6edf3")
  .attr("font-size", d => d.group === "hub" ? "11px" : "9px")
  .attr("opacity", d => d.group === "leaf" ? 0.6 : 0.9);

// Tooltip
const tooltip = d3.select("#tooltip");
node.on("mouseover", (e, d) => {{
  tooltip.style("opacity", 1)
    .html(`<strong>${{d.id}}</strong><br>Volume: ${{d.volume.toLocaleString()}} flussi<br>Tipo: ${{d.group}}`);
}}).on("mousemove", e => {{
  tooltip.style("left", (e.pageX+12)+"px").style("top", (e.pageY-10)+"px");
}}).on("mouseout", () => tooltip.style("opacity", 0));

link.on("mouseover", (e, d) => {{
  tooltip.style("opacity", 1)
    .html(`${{d.source.id}} → ${{d.target.id}}<br>${{d.weight.toLocaleString()}} flussi`);
}}).on("mousemove", e => {{
  tooltip.style("left", (e.pageX+12)+"px").style("top", (e.pageY-10)+"px");
}}).on("mouseout", () => tooltip.style("opacity", 0));

simulation.on("tick", () => {{
  link.attr("x1", d => d.source.x).attr("y1", d => d.source.y)
      .attr("x2", d => d.target.x).attr("y2", d => d.target.y);
  node.attr("transform", d => `translate(${{d.x}},${{d.y}})`);
}});

function dragstarted(e, d) {{ if(!e.active) simulation.alphaTarget(0.3).restart(); d.fx=d.x; d.fy=d.y; }}
function dragged(e, d) {{ d.fx=e.x; d.fy=e.y; }}
function dragended(e, d) {{ if(!e.active) simulation.alphaTarget(0); d.fx=null; d.fy=null; }}
</script></body></html>"""

out_path = OUTPUT_DIR / "view2_weighted.html"
out_path.write_text(html_view2, encoding="utf-8")
print(f"→ {out_path}")
print(f"  Nodi: {len(nodes_data)}, Archi: {len(links_data)}")
print(f"  Max peso arco: {max_weight:,} flussi")
print(f"  Rendering: D3.js v7 force-directed (SVG puro, no vis.js)")

G2: 42 nodi, 78 archi
→ output\view2_weighted.html
  Nodi: 42, Archi: 78
  Max peso arco: 8,556 flussi
  Rendering: D3.js v7 force-directed (SVG puro, no vis.js)


## Cella 8 — Vista 3: Solo destinazioni esterne (C2/Exfiltration focus)

### Definizione formale
Costruiamo un **sottografo** $G_3 \subseteq G_2$ applicando un filtro:

$$G_3 = \{(u, v, w) \in G_2 \mid u \in \text{RFC1918} \wedge v \notin \text{RFC1918}\}$$

Dove RFC 1918 definisce gli spazi di indirizzamento privati:
- `10.0.0.0/8`
- `172.16.0.0/12`  
- `192.168.0.0/16`

### Razionale di sicurezza
Questa vista isola i flussi **interno → esterno**, che in un contesto di incident response rappresentano:
- **Command & Control (C2):** l'host infetto "chiama casa" verso l'infrastruttura dell'attaccante
- **Data Exfiltration:** trasferimento di dati rubati verso server controllati dall'avversario
- **Beaconing:** comunicazioni periodiche a bassa entropia tipiche di RAT/backdoor

### Classificazione visuale
I nodi esterni sono classificati in due categorie con **colori nettamente distinti**:
| Colore | Categoria | Criterio |
|--------|-----------|----------|
| 🟢 Verde | Host interno (sorgente) | IP ∈ RFC 1918 |
| 🔴 Rosso | Destinazione ad **alto volume** | Flussi ≥ 65° percentile |
| 🟣 Viola | Destinazione a **basso volume** | Flussi < 65° percentile |

### Cosa cercare nell'output
- Il nodo rosso **più grande** è il primo candidato C2/exfil da investigare
- I nodi viola sono rumore di fondo (CDN, DNS, servizi legittimi) — ma vanno comunque validati

In [27]:
# Solo RFC1918 conta come "interno"
RFC1918 = [ipaddress.ip_network(n) for n in
           ("10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16")]

def is_internal(ip):
    try:
        addr = ipaddress.ip_address(ip)
        return any(addr in net for net in RFC1918)
    except ValueError:
        return False

ext = weighted[
    weighted["src"].apply(is_internal) &
    ~weighted["dst"].apply(is_internal)
]
print(f"Host interni che parlano verso l'esterno: {ext['src'].nunique()}")
print(f"Destinazioni esterne distinte:            {ext['dst'].nunique()}")

# ─── Preparazione dati per D3.js radial layout ───
max_ext_weight = ext["weight"].max()
internal_hosts = ext["src"].unique().tolist()
external_hosts = ext["dst"].unique().tolist()

# Soglia per distinguere alto/basso volume (mediana log-normalizzata)
ext_volumes = ext.groupby("dst")["weight"].sum()
volume_threshold = ext_volumes.quantile(0.65)  # sopra il 65° percentile = "alto"

nodes_v3 = []
for ip in internal_hosts:
    nodes_v3.append({"id": ip, "type": "internal", "volume": int(ext[ext["src"]==ip]["weight"].sum())})
for ip in external_hosts:
    w = int(ext[ext["dst"]==ip]["weight"].sum())
    norm_w = math.log1p(w) / math.log1p(max_ext_weight)
    # Classificazione netta: alto vs basso
    category = "high" if w >= volume_threshold else "low"
    nodes_v3.append({"id": ip, "type": "external", "volume": w, "norm": round(norm_w, 4), "category": category})

links_v3 = []
for _, row in ext.iterrows():
    w = int(row["weight"])
    norm_w = math.log1p(w) / math.log1p(max_ext_weight)
    links_v3.append({
        "source": row["src"], "target": row["dst"],
        "weight": w, "norm": round(norm_w, 4)
    })

graph_v3_json = json.dumps({"nodes": nodes_v3, "links": links_v3})

# ─── Top destinazioni per il report nel tooltip ───
top5_dst = ext.groupby("dst")["weight"].sum().nlargest(5)
top5_html = "".join(f"<br>&nbsp;&nbsp;{ip}: {w:,}" for ip, w in top5_dst.items())

n_high = sum(1 for n in nodes_v3 if n.get("category") == "high")
n_low = sum(1 for n in nodes_v3 if n.get("category") == "low")

html_view3 = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>Vista 3 — Destinazioni Esterne (potenziali C2/Exfil)</title>
<script src="https://d3js.org/d3.v7.min.js"></script>
<style>
  body {{ margin: 0; background: #0d1117; overflow: hidden; font-family: 'JetBrains Mono', 'Fira Code', monospace; }}
  svg {{ display: block; }}
  .tooltip {{
    position: absolute; padding: 8px 12px; background: rgba(13,17,23,0.96);
    border: 1px solid #30363d; border-radius: 6px; color: #e6edf3;
    font-size: 11px; pointer-events: none; opacity: 0; transition: opacity 0.2s;
    box-shadow: 0 4px 12px rgba(0,0,0,0.5); max-width: 280px;
  }}
  .legend {{
    position: absolute; top: 16px; left: 16px; background: rgba(13,17,23,0.92);
    border: 1px solid #30363d; border-radius: 8px; padding: 14px 18px;
    color: #e6edf3; font-size: 12px;
  }}
  .legend h3 {{ margin: 0 0 10px 0; font-size: 14px; color: #f85149; }}
  .legend-item {{ display: flex; align-items: center; margin: 6px 0; }}
  .legend-dot {{ width: 14px; height: 14px; border-radius: 50%; margin-right: 8px; }}
  .legend-line {{ width: 24px; height: 3px; margin-right: 8px; border-radius: 2px; }}
  .stats {{
    position: absolute; bottom: 16px; right: 16px; background: rgba(13,17,23,0.92);
    border: 1px solid #30363d; border-radius: 8px; padding: 12px 16px;
    color: #8b949e; font-size: 11px; text-align: right;
  }}
  .stats strong {{ color: #f85149; }}
  .alert-badge {{
    position: absolute; top: 16px; right: 16px; background: rgba(248,81,73,0.15);
    border: 1px solid #f85149; border-radius: 8px; padding: 10px 16px;
    color: #f85149; font-size: 12px; font-weight: bold;
  }}
</style></head><body>
<div class="legend">
  <h3>Vista 3 — External Destinations</h3>
  <div class="legend-item"><div class="legend-dot" style="background:#3fb950; border: 2px solid #2ea043;"></div>Host interno (sorgente)</div>
  <div class="legend-item"><div class="legend-dot" style="background:#f85149; border: 2px solid #da3633;"></div>Destinazione esterna — VOLUME ALTO ({n_high})</div>
  <div class="legend-item"><div class="legend-dot" style="background:#8957e5; border: 2px solid #6e40c9;"></div>Destinazione esterna — volume basso ({n_low})</div>
  <div style="margin-top:10px; border-top: 1px solid #30363d; padding-top: 8px;">
    <div class="legend-item"><div class="legend-line" style="background:#f85149;"></div>Arco pesante (molti flussi)</div>
    <div class="legend-item"><div class="legend-line" style="background:#8957e5;"></div>Arco leggero</div>
  </div>
  <div style="margin-top:8px; color:#8b949e; font-size:10px;">Soglia alto/basso: {int(volume_threshold):,} flussi</div>
</div>
<div class="alert-badge">
  ⚠ {ext['dst'].nunique()} destinazioni esterne<br>
  <span style="font-size:10px; font-weight:normal; color:#f0883e;">Top target: {top5_dst.index[0]} ({top5_dst.iloc[0]:,} flussi)</span>
</div>
<div class="stats">
  Host interni: <strong>{ext['src'].nunique()}</strong><br>
  Destinazioni: <strong>{ext['dst'].nunique()}</strong><br>
  Flusso max: <strong>{max_ext_weight:,}</strong> pacchetti<br>
  <span style="color:#58a6ff;">Top 5:{top5_html}</span>
</div>
<div class="tooltip" id="tooltip"></div>
<script>
const data = {graph_v3_json};
const width = window.innerWidth, height = window.innerHeight;
const cx = width / 2, cy = height / 2;

// Colori nettamente distinti
const COLOR_INTERNAL = "#3fb950";
const COLOR_HIGH = "#f85149";     // rosso acceso — pericolo
const COLOR_LOW = "#8957e5";      // viola — basso rischio
const STROKE_INTERNAL = "#2ea043";
const STROKE_HIGH = "#da3633";
const STROKE_LOW = "#6e40c9";

const svg = d3.select("body").append("svg").attr("width", width).attr("height", height);

// Glow filter per hub interno
const defs = svg.append("defs");
const glow = defs.append("filter").attr("id", "glow");
glow.append("feGaussianBlur").attr("stdDeviation", "4").attr("result", "coloredBlur");
const feMerge = glow.append("feMerge");
feMerge.append("feMergeNode").attr("in", "coloredBlur");
feMerge.append("feMergeNode").attr("in", "SourceGraphic");

// Arrow markers — diversi per alto/basso
defs.append("marker").attr("id", "arrow-high")
  .attr("viewBox", "0 -3 6 6").attr("refX", 16).attr("refY", 0)
  .attr("markerWidth", 4).attr("markerHeight", 4).attr("orient", "auto")
  .append("path").attr("d", "M0,-2.5L6,0L0,2.5").attr("fill", COLOR_HIGH).attr("opacity", 0.8);
defs.append("marker").attr("id", "arrow-low")
  .attr("viewBox", "0 -3 6 6").attr("refX", 16).attr("refY", 0)
  .attr("markerWidth", 4).attr("markerHeight", 4).attr("orient", "auto")
  .append("path").attr("d", "M0,-2.5L6,0L0,2.5").attr("fill", COLOR_LOW).attr("opacity", 0.6);

// Scale
const widthScale = d3.scalePow().exponent(0.4).domain([0, 1]).range([0.4, 3.5]);
const opacityScale = d3.scalePow().exponent(0.6).domain([0, 1]).range([0.25, 0.9]);
const extRadiusScale = d3.scaleSqrt().domain([0, 1]).range([6, 20]);

// Lookup: target -> category
const catLookup = {{}};
data.nodes.forEach(n => {{ if (n.category) catLookup[n.id] = n.category; }});

// Force simulation
const simulation = d3.forceSimulation(data.nodes)
  .force("link", d3.forceLink(data.links).id(d => d.id).distance(d => {{
    return d.norm > 0.5 ? 130 : 190;
  }}))
  .force("charge", d3.forceManyBody().strength(d => d.type === "internal" ? -600 : -100))
  .force("center", d3.forceCenter(cx, cy))
  .force("collision", d3.forceCollide().radius(d => d.type === "internal" ? 45 : extRadiusScale(d.norm || 0) + 10))
  .force("radial", d3.forceRadial(d => d.type === "internal" ? 0 : 250, cx, cy).strength(d => d.type === "internal" ? 0.9 : 0.25));

// Archi — colore diverso per alto/basso
const link = svg.append("g").selectAll("line").data(data.links).join("line")
  .attr("stroke", d => {{
    const cat = catLookup[d.target.id || d.target];
    return cat === "high" ? COLOR_HIGH : COLOR_LOW;
  }})
  .attr("stroke-width", d => widthScale(d.norm))
  .attr("stroke-opacity", d => opacityScale(d.norm))
  .attr("marker-end", d => {{
    const cat = catLookup[d.target.id || d.target];
    return cat === "high" ? "url(#arrow-high)" : "url(#arrow-low)";
  }});

// Nodi
const node = svg.append("g").selectAll("g").data(data.nodes).join("g")
  .call(d3.drag().on("start", dragstarted).on("drag", dragged).on("end", dragended));

// Cerchio nodo — colori netti per categoria
node.append("circle")
  .attr("r", d => {{
    if (d.type === "internal") return 24;
    return extRadiusScale(d.norm || 0);
  }})
  .attr("fill", d => {{
    if (d.type === "internal") return COLOR_INTERNAL;
    return d.category === "high" ? COLOR_HIGH : COLOR_LOW;
  }})
  .attr("stroke", d => {{
    if (d.type === "internal") return STROKE_INTERNAL;
    return d.category === "high" ? STROKE_HIGH : STROKE_LOW;
  }})
  .attr("stroke-width", d => d.type === "internal" ? 3 : 1.5)
  .attr("filter", d => d.type === "internal" ? "url(#glow)" : null)
  .attr("opacity", d => {{
    if (d.type === "internal") return 1;
    return d.category === "high" ? 0.95 : 0.7;
  }});

// Label
node.append("text")
  .text(d => d.id)
  .attr("dx", d => d.type === "internal" ? 0 : (extRadiusScale(d.norm||0) + 5))
  .attr("dy", d => d.type === "internal" ? -30 : 3)
  .attr("text-anchor", d => d.type === "internal" ? "middle" : "start")
  .attr("fill", d => {{
    if (d.type === "internal") return COLOR_INTERNAL;
    return d.category === "high" ? "#ffa198" : "#d2a8ff";
  }})
  .attr("font-size", d => {{
    if (d.type === "internal") return "12px";
    return d.category === "high" ? "10px" : "9px";
  }})
  .attr("font-weight", d => (d.type === "internal" || d.category === "high") ? "bold" : "normal")
  .attr("opacity", d => {{
    if (d.type === "internal") return 1;
    return d.category === "high" ? 1 : 0.6;
  }});

// Etichetta "INFECTED HOST" per nodo interno
node.filter(d => d.type === "internal").append("text")
  .text("INFECTED HOST")
  .attr("dy", -44).attr("text-anchor", "middle")
  .attr("fill", "#f0883e").attr("font-size", "10px").attr("font-weight", "bold");

// Tooltip
const tooltip = d3.select("#tooltip");
node.on("mouseover", (e, d) => {{
  let info;
  if (d.type === "internal") {{
    info = `<strong style="color:${{COLOR_INTERNAL}}">${{d.id}}</strong><br>Tipo: Host interno (infetto)<br>Flussi verso l'esterno: ${{d.volume.toLocaleString()}}`;
  }} else {{
    const col = d.category === "high" ? COLOR_HIGH : COLOR_LOW;
    const label = d.category === "high" ? "ALTO VOLUME" : "basso volume";
    info = `<strong style="color:${{col}}">${{d.id}}</strong><br>Tipo: Destinazione esterna (${{label}})<br>Flussi ricevuti: ${{d.volume.toLocaleString()}}`;
  }}
  tooltip.style("opacity", 1).html(info);
}}).on("mousemove", e => {{
  tooltip.style("left", (e.pageX+12)+"px").style("top", (e.pageY-10)+"px");
}}).on("mouseout", () => tooltip.style("opacity", 0));

link.on("mouseover", (e, d) => {{
  tooltip.style("opacity", 1)
    .html(`${{d.source.id}} &rarr; ${{d.target.id}}<br><strong>${{d.weight.toLocaleString()}}</strong> flussi`);
}}).on("mousemove", e => {{
  tooltip.style("left", (e.pageX+12)+"px").style("top", (e.pageY-10)+"px");
}}).on("mouseout", () => tooltip.style("opacity", 0));

// Highlight su hover nodo
node.on("mouseenter", function(e, d) {{
  const connected = new Set();
  data.links.forEach(l => {{
    if (l.source.id === d.id) connected.add(l.target.id);
    if (l.target.id === d.id) connected.add(l.source.id);
  }});
  connected.add(d.id);
  node.attr("opacity", n => connected.has(n.id) ? 1 : 0.1);
  link.attr("opacity", l => (l.source.id === d.id || l.target.id === d.id) ? 1 : 0.03);
}}).on("mouseleave", () => {{
  node.attr("opacity", 1);
  link.attr("opacity", 1);
}});

simulation.on("tick", () => {{
  link.attr("x1", d => d.source.x).attr("y1", d => d.source.y)
      .attr("x2", d => d.target.x).attr("y2", d => d.target.y);
  node.attr("transform", d => `translate(${{d.x}},${{d.y}})`);
}});

function dragstarted(e, d) {{ if(!e.active) simulation.alphaTarget(0.3).restart(); d.fx=d.x; d.fy=d.y; }}
function dragged(e, d) {{ d.fx=e.x; d.fy=e.y; }}
function dragended(e, d) {{ if(!e.active) simulation.alphaTarget(0); d.fx=null; d.fy=null; }}
</script></body></html>"""

out_path3 = OUTPUT_DIR / "view3_external.html"
out_path3.write_text(html_view3, encoding="utf-8")
print(f"-> {out_path3}")
print(f"  Host interni: {ext['src'].nunique()}, Destinazioni: {ext['dst'].nunique()}")
print(f"  Volume alto (rosso): {n_high} nodi | Volume basso (viola): {n_low} nodi")
print(f"  Soglia: {int(volume_threshold):,} flussi")
print(f"  Arco piu pesante: {max_ext_weight:,} flussi")
print(f"\n  Top 5 destinazioni per volume:")
for ip, w in top5_dst.items():
    print(f"    {ip:<22} {w:>6,} flussi")

Host interni che parlano verso l'esterno: 1
Destinazioni esterne distinte:            39
-> output\view3_external.html
  Host interni: 1, Destinazioni: 39
  Volume alto (rosso): 14 nodi | Volume basso (viola): 25 nodi
  Soglia: 38 flussi
  Arco piu pesante: 8,556 flussi

  Top 5 destinazioni per volume:
    165.245.215.18          8,556 flussi
    172.67.165.232            262 flussi
    208.54.62.240             217 flussi
    172.67.198.113            197 flussi
    142.251.116.94            194 flussi


## Celle 9–11 — Bonus: Algoritmi di Graph Theory applicati live

### Collegamento con la lezione teorica T2
Nella parte teorica abbiamo studiato gli algoritmi fondamentali della teoria dei grafi in astratto. In questa sezione li **applichiamo al grafo $G_2$ costruito dal pcap reale**, dimostrando che:

1. Gli algoritmi non sono solo esercizi matematici, ma strumenti operativi
2. La stessa struttura dati (il grafo) supporta analisi multiple
3. I risultati sono **interpretabili nel dominio della cybersecurity**

### Algoritmi applicati

| Algoritmo | Complessità | Cosa rivela nel nostro contesto |
|-----------|-------------|-------------------------------|
| **PageRank** | $O(\|V\| + \|E\|)$ per iterazione | Centralità "reputazionale": chi riceve più traffico *indirettamente* |
| **Connected Components** | $O(\|V\| + \|E\|)$ | Cluster isolati di comunicazione — topologie separate |
| **Visualizzazione PageRank** | — | Mappa calore della centralità sovrapposta al layout del grafo |

### Cella 9 — PageRank: centralità nel traffico di rete

**PageRank** (Brin & Page, 1998) assegna a ogni nodo un punteggio proporzionale alla probabilità che un "random surfer" lo visiti, seguendo gli archi pesati. Nel nostro contesto:

$$PR(v) = \frac{1-d}{|V|} + d \sum_{u \in \text{in}(v)} \frac{w(u,v) \cdot PR(u)}{\sum_{k \in \text{out}(u)} w(u,k)}$$

dove $d = 0.85$ è il damping factor.

**Interpretazione operativa:**
- Un nodo con PageRank alto è un **hub di comunicazione** — molti flussi convergono verso di esso direttamente o indirettamente
- In un'infezione: l'host infetto avrà PageRank dominante perché **genera** la quasi totalità degli archi uscenti
- Un server C2 esterno potrebbe emergere se riceve traffico da più host interni compromessi

In [28]:
pr = nx.pagerank(G2, weight="weight")
top10 = sorted(pr.items(), key=lambda x: -x[1])[:10]

print("Top 10 nodi per PageRank:\n")
print(f"{'IP':<20} {'Score':<10} {'Tipo'}")
print("-" * 45)
for ip, score in top10:
    tipo = "interno" if is_internal(ip) else "ESTERNO"
    print(f"{ip:<20} {score:<10.4f} {tipo}")

Top 10 nodi per PageRank:

IP                   Score      Tipo
---------------------------------------------
10.5.11.101          0.4542     interno
165.245.215.18       0.3182     ESTERNO
172.67.165.232       0.0135     ESTERNO
208.54.62.240        0.0119     ESTERNO
172.67.198.113       0.0111     ESTERNO
142.251.116.94       0.0110     ESTERNO
94.232.249.129       0.0085     ESTERNO
17.253.126.69        0.0072     ESTERNO
10.5.11.1            0.0071     interno
142.250.114.113      0.0071     ESTERNO


### Cella 10 — Componenti connesse: segmentazione topologica

Una **componente connessa** è un sottografo massimale in cui esiste un cammino tra ogni coppia di nodi (considerando il grafo come non diretto).

**Interpretazione operativa:**
- Se il grafo ha **una sola componente** → tutto il traffico è interconnesso (tipico di una LAN con gateway unico)
- **Componenti multiple** → esistono segmenti di rete isolati, oppure host che comunicano solo tra loro (possibile lateral movement contenuto)
- In reti aziendali segmentate correttamente, ci aspettiamo più componenti (DMZ, VLAN separate, etc.)

> **Nota:** nella teoria dei grafi, il numero di componenti connesse è il primo invariante topologico da calcolare — ci dice se il grafo è "un oggetto unico" o un insieme disconnesso.

In [29]:
G2_undirected = G2.to_undirected()
components = list(nx.connected_components(G2_undirected))
components.sort(key=len, reverse=True)

print(f"Numero di componenti connesse: {len(components)}\n")
for i, comp in enumerate(components[:5], 1):
    print(f"Componente #{i}: {len(comp)} nodi  — esempio: {list(comp)[:3]}")

Numero di componenti connesse: 1

Componente #1: 42 nodi  — esempio: ['17.253.12.37', '173.194.208.94', '104.21.26.126']


### Cella 11 — Visualizzazione con PageRank come mappa di calore

In questa vista finale, il **colore di ogni nodo codifica il suo PageRank score**:
- Rosso intenso = alta centralità (PageRank elevato)
- Colore tenue = nodo periferico

Questo tipo di encoding è equivalente a una **heatmap sovrapposta alla topologia del grafo** — tecnica comune in network security monitoring per identificare rapidamente anomalie strutturali.

> **Esercizio per lo studente:** confrontare questa vista con la Vista 3. L'host con il PageRank massimo è lo stesso che genera tutti i flussi esterni? Perché?

In [30]:
# Color encoding: PageRank → intensità
pr_max = max(pr.values()) if pr else 1
G_final = G2.copy()
for n in G_final.nodes():
    score = pr.get(n, 0) / pr_max
    # rosso più intenso = PageRank più alto
    red = int(80 + 175 * score)
    G_final.nodes[n]["color"] = f"rgb({red},{100 - int(50*score)},{100 - int(50*score)})"
    G_final.nodes[n]["title"] = f"{n}\nPageRank: {pr.get(n, 0):.4f}"

net_final = Network(notebook=False, directed=True, height="700px", width="100%",
                    bgcolor="#1a1a1a", font_color="white")
net_final.from_nx(G_final)
net_final.write_html(str(OUTPUT_DIR / "view_bonus_pagerank.html"),
                     open_browser=False, notebook=False)
print(f"→ {OUTPUT_DIR / 'view_bonus_pagerank.html'}")

→ output\view_bonus_pagerank.html


## Riepilogo — Concetti dimostrati

### Competenze acquisite in questa demo

| # | Concetto | Implementazione |
|---|----------|----------------|
| 1 | **PCAP → Grafo** | Da una cattura di traffico reale a un modello matematico interrogabile, in <30 righe Python |
| 2 | **Viste progressive** | Tre rappresentazioni dello stesso oggetto $G$: naive, pesata, filtrata — ognuna rivela struttura diversa |
| 3 | **Algoritmi su dati reali** | PageRank e connected components applicati a traffico di rete (non a grafi giocattolo) |
| 4 | **Visual encoding** | Mapping variabile→proprietà visiva (dimensione, colore, spessore) secondo i principi di Bertin (1967) |

### Bridge verso la demo successiva (BloodHound)
Lo **stesso paradigma** (entità → nodi, relazioni → archi, proprietà → pesi) si applica al dominio delle **identità Active Directory**:
- Nodi: utenti, gruppi, computer, GPO
- Archi: `MemberOf`, `HasSession`, `AdminTo`, `CanRDP`
- Algoritmo chiave: *shortest path to Domain Admin*

La struttura matematica è identica; cambia solo il dominio semantico.

### Riferimenti bibliografici
- Brin, S. & Page, L. (1998). *The Anatomy of a Large-Scale Hypertextual Web Search Engine*. Stanford University.
- Bertin, J. (1967). *Sémiologie Graphique*. Gauthier-Villars.
- MITRE ATT&CK — T1071 (Application Layer Protocol), T1041 (Exfiltration Over C2 Channel)